<div style="width:100%; text-align:center; padding:10px 0;">
<img src="project_header.png" style="width:100%; max-width:100vw; height:auto; display:block; margin:0 auto;">
</div>

# EO Africa SWAM project 
## Create list of Sentinel-2 L1C filenames

#### This notebook does the following:
* use EODAG to search for Sentinel-2 L1C files for a region of interest and date range
* save all the filenames between given dates (default is 1 Jan 2015 - 31 Dec 2025) as a text file

#### Settings to adjust manually:
* in code cell 3 uncomment the lake/dam name and polygon (options: ZV, RV, TW, MV, VV, CW), default is TW.
* in code cell 4 ensure that the appropriate start and end dates are defined in format "yyyy-mm-dd"

##### Authors:
**Marie Smith**, CSIR, South Africa

In [ ]:
import os
import glob
import subprocess
import folium
import imageio
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from eodag import EODataAccessGateway, setup_logging
import re

First define your functions. For some dates there might be more than one data processing version available (e.g. N0509, N0510), so we are defining a fuction that screens all the filenames and only keeps the one with the highest (latest) processing version.

In [ ]:
def filter_latest_processing_version(file_list):
    """
    Filters Sentinel-2 L1C files to keep only the most recent processing version per sensing date.
    Parameters:
    - file_list: List of Sentinel-2 L1C file paths (strings)
    Returns:
    - List of filtered file paths with only the latest processing version per sensing date
    """
    pattern = re.compile(r'/S2[ABC]_MSIL1C_(\d{8}T\d{6})_N(\d{4})_R\d{3}_T\w+_(\d{8}T\d{6})\.SAFE/')
    
    latest_files = {}
    
    for file_path in file_list:
        match = pattern.search(file_path)
        if match:
            sensing_date = match.group(1)  # e.g., 20231002T081749
            processing_version = match.group(2)  # e.g., 0509, 0510
            # Use sensing_date as key to group same date files
            if sensing_date not in latest_files:
                latest_files[sensing_date] = (processing_version, file_path)
            else:
                # Keep file with the higher (newer) processing version number
                if processing_version > latest_files[sensing_date][0]:
                    latest_files[sensing_date] = (processing_version, file_path)
    
    # Extract the filtered file paths
    filtered_files = [file_info[1] for file_info in latest_files.values()]
    return filtered_files

Next we are providing the name of the lake for filename purposes, and providing the polygon over which the EODAG tool will search for the data. MOST IMPORTANT - Unhash the specific lake_name and geo_search polygon that you want to use below

In [ ]:
## Voelvlei dam:
#lake_name = "VV"
#geo_search = "POLYGON ((19.018662 -33.406898, 19.018662 -33.33235, 19.066678 -33.33235, 19.066678 -33.406898, 19.018662 -33.406898))"

## Misverstand dam:
#lake_name = "MV"
#geo_search = "POLYGON ((18.786192 -33.033152, 18.786192 -33.022862, 18.81855 -33.022862, 18.81855 -33.033152, 18.786192 -33.033152))"

## Clanwilliam dam (DON'T USE THIS POLYGON FOR SUBSETTING, ONLY FOR SEARCHING CORRECT TILES!):
#lake_name = "CW"
#geo_search = "POLYGON ((18.860779 -32.302804, 18.860779 -32.287133, 18.869705 -32.287133, 18.869705 -32.302804, 18.860779 -32.302804))"

## Theewaterskloof dam:
lake_name = "TW"
geo_search = "POLYGON ((19.0989 -33.9652, 19.0989 -34.0982, 19.3181 -34.0982, 19.3181 -33.9652, 19.0989 -33.9652))"

## Zeekoevlei:
#lake_name = "ZV"
#geo_search = "POLYGON ((18.50235 -34.073991, 18.50235 -34.048108, 18.522606 -34.048108, 18.522606 -34.073991, 18.50235 -34.073991))"

## Rietvlei:
#lake_name = "RV"
#geo_search = "POLYGON ((18.476077 -33.856497, 18.476077 -33.825699, 18.519984 -33.825699, 18.519984 -33.856497, 18.476077 -33.856497))"


The EODAG tool can give a timeout error if you try to search for too many files at once. So we search for files in batches of one year, and then append each batch of results to the filename list. If you want to use a shorter time period, you can just input a single start date and end date in format "yyyy-mm-dd"

In [ ]:
start_date_list = ["2015-01-01","2016-01-01","2017-01-01","2018-01-01","2019-01-01","2020-01-01","2021-01-01","2022-01-01","2023-01-01","2024-01-01","2025-01-01"]
end_date_list = ["2015-12-31","2016-12-31","2017-12-31","2018-12-31","2019-12-31","2020-12-31","2021-12-31","2022-12-31","2023-12-31","2024-12-31","2025-12-31"]
filtered_list = []
list_length = len(start_date_list)

Now we will use the EODAG tool to search for all the filenames that intersect with our search area and time period within the *eodata* folder of the Jupyter Lab, and create a list of the precise file locations. You can either use **creodias** or **cop_dataspace** as the provider

In [ ]:
## for "provider" you can try either creodias or cop_dataspace
for ii in range(list_length):
    dag = EODataAccessGateway()
    geometry = geo_search
    search_results = dag.search_all(
        provider="creodias",
        productType="S2_MSI_L1C",
        geom=geometry,
        start=start_date_list[ii],
        end=end_date_list[ii],
    )
    num = len(search_results)
    print(str(num) + ' files found in round '+str(ii+1))

    all_image = []
    for file in search_results:
        file = file.properties['productIdentifier']
        all_image.append('/home/eoafrica'+file+'/MTD_MSIL1C.xml')

    ## filter out the data with the same dates so that only the most recent processing version remains:
    filtered_files = filter_latest_processing_version(all_image)
    fnum = len(filtered_files)
    print(str(fnum) + ' filtered files left in round '+str(ii+1))
    filtered_list.extend(filtered_files)

tnum = len(filtered_list)
print(str(tnum) + ' filtered files total')

In [ ]:
## save this output to a text file for future use:
with open(lake_name+'_filenames.txt', 'w') as file:
    for item in filtered_list:
        file.write(f"{item}\n")

You should now have a text file with all of the file locations that you can use in *Chl_Step2_ACOLITE-RAdCor.ipynb* or *TSM_Step2_C2XComplex.ipynb*